<a href="https://colab.research.google.com/github/Abhishek2993/BrainCancer-Detection-MRI-DATA-/blob/main/Phase2_Edge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCDC Phase 2: Edge-efficient, trustworthy diagnosis

Can a brain-tumour classifier run on cheap hardware without losing accuracy or **trustworthiness** (calibration and conformal coverage)? Can hospitals train it together **without sharing scans**?

| Section | What it does |
|---|---|
| **A. Backbone benchmark** | Trains MobileNetV2, MobileNetV3-Small/Large, EfficientNetB0 and ConvNeXt-Tiny with the same recipe and the same split. |
| **B. Quantisation** | Exports each model to TensorFlow Lite in FP32, FP16, dynamic-range and full-INT8 precision. |
| **C. Trade-offs** | Accuracy, glioma sensitivity, calibration (ECE), conformal coverage and set size, size and latency for every variant, plus the Pareto front. It also tests whether conformal thresholds survive quantisation or must be recalibrated. |
| **D. Edge bundle** | Packs the models, a test set and the calibration settings, with a script that benchmarks them on a **Raspberry Pi / Jetson**. |
| **E. Federated learning** | Simulates 4 hospitals: centralised vs FedAvg vs FedProx vs each-hospital-alone, with similar (IID) and different (non-IID) case mixes. |

**Before running:** run the Phase 0 notebook. This notebook uses its split, and reuses its MobileNetV2 seed-0 model, from Google Drive (`MyDrive/BCDC/outputs`). Use a **GPU runtime** for training. Latency is measured on the Colab **CPU**, as a stand-in for edge CPUs; real device numbers come from section D.

Every trained model and result is saved to Drive as it's produced. If Colab disconnects, re-running skips the work already done.

> ⚠️ Research and educational use only. Not a medical device.

## 0. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Abhishek2993/BrainCancer-Detection-MRI-DATA-.git"
REPO_DIR = Path(os.environ.get("BCDC_REPO_DIR", "BrainCancer-Detection-MRI-DATA-"))
try:
    import bcdc
except ImportError:
    if not (REPO_DIR / "bcdc").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    sys.path.insert(0, str(REPO_DIR.resolve()))
    import bcdc
print("bcdc", bcdc.__version__, "from", Path(bcdc.__file__).parent)

try:
    import ai_edge_litert  # noqa: F401  (LiteRT interpreter; falls back to tf.lite if missing)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ai-edge-litert"], check=False)

In [ ]:
import json
import shutil
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import tensorflow as tf
from tensorflow import keras

from bcdc.backbones import BACKBONES, build_classifier, set_finetune_trainable
from bcdc.conformal import ConformalClassifier, set_metrics
from bcdc.data import display_name, find_class_root, labeled_path_dataset, load_split, split_validation
from bcdc.edge import QUANT_MODES, TFLiteClassifier, keras_latency_ms, pareto_front, to_tflite
from bcdc.federated import FederatedTrainer, dirichlet_partition, iid_partition
from bcdc.metrics import apply_temperature, evaluate, expected_calibration_error, fit_temperature
from bcdc.train import train_two_stage

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---- Configuration -------------------------------------------------------
BACKBONE_NAMES = ["mobilenet_v2", "mobilenet_v3_small", "mobilenet_v3_large", "efficientnet_b0", "convnext_tiny"]
SEED = 0                      # one training seed per backbone (Phase 0 measured seed-to-seed spread for MobileNetV2)
BATCH_SIZE = 32
HEAD_EPOCHS, HEAD_LR = 15, 1e-3
FINETUNE_EPOCHS, FINETUNE_LR = 20, 1e-5
TOP_FRACTION = 0.35           # share of backbone layers unfrozen in stage 2 (MobileNetV2: layers >= 100)
PATIENCE = 4

ALPHA = 0.10                  # conformal target: 90% coverage, class-conditional LAC
LATENCY_RUNS = 50
LATENCY_THREADS = [1, 4]      # CPU threads for TFLite latency (Raspberry Pi 4/5 have 4 cores)
N_CALIBRATION = 200           # representative images for INT8 calibration (from the training set)
EDGE_TESTSET_MAX = 300        # test images shipped in the edge bundle
LATENCY_BUDGET_MS = 100       # selection rule: most accurate (on val-A) variant under this 1-thread latency

# Federated learning
FL_BACKBONE = "mobilenet_v2"
FL_CLIENTS = 4
FL_ROUNDS = 15
FL_LOCAL_EPOCHS = 1
FL_LR = 1e-4
FL_DIRICHLET_ALPHA = 0.5      # non-IID label skew (smaller = more different hospitals)
FL_PROX_MU = 0.01

if os.environ.get("BCDC_QUICK_RUN") == "1":   # offline smoke tests only
    BACKBONE_NAMES = ["mobilenet_v2", "mobilenet_v3_small", "efficientnet_b0"]
    HEAD_EPOCHS, FINETUNE_EPOCHS, LATENCY_RUNS, N_CALIBRATION = 2, 1, 5, 20
    FL_CLIENTS, FL_ROUNDS, LATENCY_THREADS, EDGE_TESTSET_MAX = 3, 3, [1], 20
WEIGHTS = None if os.environ.get("BCDC_WEIGHTS") == "none" else "imagenet"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if os.environ.get("BCDC_ARTEFACT_DIR"):
    ARTEFACT_DIR = Path(os.environ["BCDC_ARTEFACT_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTEFACT_DIR = Path("/content/drive/MyDrive/BCDC/outputs")
else:
    ARTEFACT_DIR = Path("outputs")

P2 = ARTEFACT_DIR / "phase2"
for d in ["models", "tflite", "figures", "federated"]:
    (P2 / d).mkdir(parents=True, exist_ok=True)

meta = json.loads((ARTEFACT_DIR / "model_meta.json").read_text())
CLASS_NAMES = meta["class_names"]
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = tuple(meta["img_size"])
G = CLASS_NAMES.index("brain_glioma")
print("Artefacts:", ARTEFACT_DIR, "| image size:", IMG_SIZE, "| backbones:", BACKBONE_NAMES)


def md_table(df, floatfmt="{:.3f}"):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join([""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx, row in df.iterrows():
        vals = [floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join([str(idx)] + vals) + " |")
    return "\n".join(lines)


REPORT = []

## 1. Data: the Phase 0 split

The split is the same as in Phase 0. The validation set is divided into val-A and val-B exactly as in Phase 1:

* **val-A:** early stopping during training, temperature fitting, and choosing the deployed model;
* **val-B:** calibrating the conformal thresholds;
* **test:** reporting only.

In [ ]:
if os.environ.get("BCDC_DATASET_PATH"):
    DATASET_PATH = find_class_root(os.environ["BCDC_DATASET_PATH"])
else:
    import kagglehub
    path = kagglehub.dataset_download("orvile/brain-cancer-mri-dataset")
    expected = Path(path) / "Brain_Cancer raw MRI data" / "Brain_Cancer"
    DATASET_PATH = expected if expected.is_dir() else find_class_root(path)

split = split_validation(load_split(ARTEFACT_DIR / "split.csv", DATASET_PATH), seed=0)
SPLITS = ["train", "valA", "valB", "test"]
Y = {s: split.loc[split["split"] == s, "label"].to_numpy() for s in SPLITS}
PATHS = {s: split.loc[split["split"] == s, "abs_path"].to_numpy() for s in SPLITS}
print(pd.crosstab(split["cls"], split["split"])[SPLITS])


def eval_ds(s):
    return labeled_path_dataset(PATHS[s], Y[s], IMG_SIZE, BATCH_SIZE)


VAL_DS = {s: eval_ds(s) for s in ["valA", "valB", "test"]}
# uint8 image arrays for TFLite evaluation, INT8 calibration and the edge bundle
ARR = {s: np.concatenate([x.numpy() for x, _ in VAL_DS[s]]).astype(np.uint8) for s in ["valA", "valB", "test"]}
rng = np.random.default_rng(SEED)
calib_idx = rng.choice(len(PATHS["train"]), min(N_CALIBRATION, len(PATHS["train"])), replace=False)
CALIB = np.concatenate([x.numpy() for x, _ in labeled_path_dataset(PATHS["train"][calib_idx], Y["train"][calib_idx],
                                                                  IMG_SIZE, BATCH_SIZE, cache=False)])

## A. Train every backbone with the same recipe

The recipe is the same as Phase 0: train the head with the backbone frozen, then fine-tune the top 35% of the backbone with normalisation layers frozen. Early stopping uses val-A.

Models are saved to Drive as they finish. The MobileNetV2 seed-0 model from Phase 0 is reused, because it was trained with exactly this recipe on this split. Phase 0 used the whole validation set for early stopping, so it saw slightly more validation data.

In [ ]:
MODELS = {}
for name in BACKBONE_NAMES:
    path = P2 / "models" / f"{name}.keras"
    phase0 = ARTEFACT_DIR / "models" / f"seed_{SEED}.keras"
    if not path.exists() and name == "mobilenet_v2" and phase0.exists():
        shutil.copy(phase0, path)
        print("mobilenet_v2: reusing the Phase 0 seed-0 model")
    if path.exists():
        MODELS[name] = keras.models.load_model(path)
        print(f"{name}: loaded from {path.name}")
        continue
    print(f"\n========== training {name} ==========")
    t0 = time.time()
    model, base = build_classifier(name, NUM_CLASSES, IMG_SIZE, weights=WEIGHTS, seed=SEED)
    train_ds = labeled_path_dataset(PATHS["train"], Y["train"], IMG_SIZE, BATCH_SIZE, shuffle=True, seed=SEED)
    train_two_stage(model, base, train_ds, VAL_DS["valA"], HEAD_EPOCHS, HEAD_LR, FINETUNE_EPOCHS, FINETUNE_LR,
                    TOP_FRACTION, PATIENCE)
    model.save(path)
    MODELS[name] = model
    print(f"{name}: trained in {(time.time() - t0) / 60:.1f} min")

## B. Export to TensorFlow Lite in four precisions

* **fp32:** a plain conversion, used as the reference.
* **fp16:** weights stored in half precision.
* **dynamic:** int8 weights, float activations.
* **int8:** full integer. The input is uint8 RGB (0–255), and the converter is calibrated on 200 training images.

The augmentation and dropout layers are stripped before export.

In [ ]:
VARIANTS = {}   # (backbone, mode) -> path
for name, model in MODELS.items():
    for mode in QUANT_MODES:
        path = P2 / "tflite" / f"{name}_{mode}.tflite"
        if not path.exists():
            t0 = time.time()
            path.write_bytes(to_tflite(model, mode, representative_images=CALIB))
            print(f"{path.name}: {path.stat().st_size / 1e6:.2f} MB [{time.time() - t0:.0f}s]")
        VARIANTS[(name, mode)] = path

## C. Evaluate every variant: accuracy, trust and speed

For each Keras model and TFLite variant:

* **Accuracy, macro-F1 and glioma sensitivity** on the test set.
* **ECE** after a temperature fitted on val-A, separately for each variant.
* **Conformal** (class-conditional LAC, 90% target): coverage and average set size, **recalibrated** on val-B for each variant. The *reused FP32 thresholds* column shows what happens if you quantise but keep the FP32 calibration.
* **Agreement with FP32:** the share of test images where the variant predicts the same class.
* **Latency:** single-image inference on the Colab CPU, 1 and 4 threads (median of 50 runs), plus file size.
* **Delegate:** whether the fast XNNPACK CPU kernels ran. Some INT8 graphs, e.g. MobileNetV3's hard-swish, fail to prepare in XNNPACK; they then fall back to slower built-in kernels, which shows up in the latency.

In [ ]:
def trust_metrics(p_valA, p_valB, p_test, cp_ref=None):
    T = fit_temperature(Y["valA"], p_valA)
    pa, pb, pt = (apply_temperature(p, T) for p in (p_valA, p_valB, p_test))
    m, _ = evaluate(Y["test"], pt, CLASS_NAMES)
    cp = ConformalClassifier(ALPHA, "lac", class_conditional=True).fit(pb, Y["valB"])
    sm = set_metrics(cp.predict_sets(pt), Y["test"])
    out = {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "glioma_sens": m[f"sensitivity[{CLASS_NAMES[G]}]"],
           "ece": m["ece"], "coverage": sm["coverage"], "set_size": sm["avg_set_size"],
           "valA_accuracy": float((pa.argmax(1) == Y["valA"]).mean()), "temperature": T}
    if cp_ref is not None:
        out["coverage_reused_fp32_thresholds"] = set_metrics(cp_ref.predict_sets(pt), Y["test"])["coverage"]
    return out, cp


rows, CAL = [], {}
results_csv = P2 / "variants.csv"
for name, model in MODELS.items():
    p = {s: model.predict(VAL_DS[s], verbose=0) for s in ["valA", "valB", "test"]}
    ref_pred = p["test"].argmax(1)
    keras_m, cp_fp32 = trust_metrics(p["valA"], p["valB"], p["test"])
    rows.append({"backbone": name, "variant": "keras", **keras_m,
                 "agree_fp32": 1.0, "size_mb": float("nan"),
                 "latency_ms": keras_latency_ms(model, n_runs=max(5, LATENCY_RUNS // 2))["median_ms"]})
    for mode in QUANT_MODES:
        clf = TFLiteClassifier(model_path=VARIANTS[(name, mode)])
        pq = {s: clf.predict(ARR[s]) for s in ["valA", "valB", "test"]}
        m, cp = trust_metrics(pq["valA"], pq["valB"], pq["test"], cp_ref=cp_fp32)
        row = {"backbone": name, "variant": mode, **m, "agree_fp32": float((pq["test"].argmax(1) == ref_pred).mean()),
               "size_mb": VARIANTS[(name, mode)].stat().st_size / 1e6, "delegate": clf.delegate}
        for th in LATENCY_THREADS:
            lat = TFLiteClassifier(model_path=VARIANTS[(name, mode)], num_threads=th).latency_ms(LATENCY_RUNS)
            row[f"latency_{th}t_ms"] = lat["median_ms"]
        row["latency_ms"] = row[f"latency_{LATENCY_THREADS[0]}t_ms"]
        rows.append(row)
        CAL[f"{name}_{mode}"] = {"temperature": m["temperature"], "qhat": cp.to_dict()["qhat"]}
        print(f"{name:<20s} {mode:<8s} acc {100 * m['accuracy']:5.1f}%  ECE {m['ece']:.3f}  "
              f"cov {m['coverage']:.3f}  {row['size_mb']:6.2f} MB  {row['latency_ms']:7.2f} ms")
variants = pd.DataFrame(rows)
variants.to_csv(results_csv, index=False)

show = variants.set_index(variants["backbone"] + " · " + variants["variant"])
cols = ["accuracy", "macro_f1", "glioma_sens", "ece", "coverage", "coverage_reused_fp32_thresholds", "set_size",
        "agree_fp32", "size_mb"] + [c for c in show.columns if c.startswith("latency_") and c.endswith("t_ms")] + ["delegate"]
table = show[[c for c in cols if c in show.columns]]
display(Markdown(f"**All variants (internal test set, {len(Y['test'])} images; latency = Colab CPU, batch 1)**\n\n" + md_table(table)))
REPORT.append("### B–C. Backbones × precision (internal test set)\n\n" + md_table(table))

In [ ]:
# ---- Trade-off plots ----
tfl = variants[variants["variant"] != "keras"].reset_index(drop=True)
tfl["pareto"] = pareto_front(tfl, maximise=("accuracy",), minimise=("latency_ms", "size_mb"))
markers = {"fp32": "o", "fp16": "s", "dynamic": "^", "int8": "D"}
colors = dict(zip(BACKBONE_NAMES, plt.cm.tab10.colors))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for _, r in tfl.iterrows():
    kw = dict(color=colors[r["backbone"]], marker=markers[r["variant"]], s=30 + 6 * r["size_mb"],
              edgecolor="black" if r["pareto"] else "none", linewidth=1.5)
    axes[0].scatter(r["latency_ms"], 100 * r["accuracy"], **kw)
    axes[1].scatter(r["latency_ms"], r["ece"], **kw)
    axes[2].scatter(r["size_mb"], r["set_size"], **kw)
axes[0].set(xscale="log", xlabel=f"latency, ms (Colab CPU, {LATENCY_THREADS[0]} thread, log)", ylabel="test accuracy (%)",
            title="Accuracy vs latency (black edge = Pareto front)")
axes[1].set(xscale="log", xlabel="latency, ms (log)", ylabel="ECE after temperature", title="Calibration vs latency")
axes[2].set(xscale="log", xlabel="model size, MB (log)", ylabel="avg conformal set size (90%)",
            title="Prediction-set size vs model size")
handles = [plt.Line2D([], [], color=c, marker="o", ls="", label=b) for b, c in colors.items() if b in MODELS]
handles += [plt.Line2D([], [], color="grey", marker=m, ls="", label=v) for v, m in markers.items()]
axes[2].legend(handles=handles, fontsize=7, loc="best")
plt.tight_layout(); plt.savefig(P2 / "figures" / "tradeoffs.png", dpi=120); plt.show()

pareto_tbl = tfl[tfl["pareto"]].set_index(tfl.loc[tfl["pareto"], "backbone"] + " · " + tfl.loc[tfl["pareto"], "variant"])
display(Markdown("**Pareto front** (accuracy ↑, latency ↓, size ↓)\n\n" +
                 md_table(pareto_tbl[["accuracy", "ece", "coverage", "set_size", "size_mb", "latency_ms"]])))
REPORT.append("**Pareto front** (accuracy ↑, latency ↓, size ↓)\n\n" +
              md_table(pareto_tbl[["accuracy", "ece", "coverage", "set_size", "size_mb", "latency_ms"]]))

# Quantisation effect on trust: INT8 vs FP32 per backbone
q = tfl.pivot(index="backbone", columns="variant", values=["accuracy", "ece", "coverage", "coverage_reused_fp32_thresholds"])
quant = pd.DataFrame({
    "Δ accuracy (int8 − fp32)": q[("accuracy", "int8")] - q[("accuracy", "fp32")],
    "Δ ECE (int8 − fp32)": q[("ece", "int8")] - q[("ece", "fp32")],
    "coverage int8, recalibrated": q[("coverage", "int8")],
    "coverage int8, reused fp32 thresholds": q[("coverage_reused_fp32_thresholds", "int8")],
})
display(Markdown("**Does INT8 quantisation break trust?**\n\n" + md_table(quant)))
REPORT.append("**INT8 vs FP32**\n\n" + md_table(quant))

# Deployment choice: made on VALIDATION accuracy (val-A), not on test results
eligible = tfl[tfl["latency_ms"] <= LATENCY_BUDGET_MS]
choice = (eligible if len(eligible) else tfl).sort_values(["valA_accuracy", "latency_ms"], ascending=[False, True]).iloc[0]
CHOSEN = f"{choice['backbone']}_{choice['variant']}"
print(f"Selected for deployment (best val-A accuracy within {LATENCY_BUDGET_MS} ms): {CHOSEN} · "
      f"test acc {100 * choice['accuracy']:.1f}%, {choice['size_mb']:.2f} MB, {choice['latency_ms']:.1f} ms")
REPORT.append(f"**Selected edge model** (best val-A accuracy within {LATENCY_BUDGET_MS} ms on 1 CPU thread): `{CHOSEN}`")

## D. Edge bundle for a Raspberry Pi / Jetson

The bundle, `bcdc_edge_bundle.zip` on Drive, contains:

* `models/`: the INT8 and FP16 TFLite model of every backbone;
* `testset.npz`: up to 300 held-out test images (uint8) with their labels;
* `calibration.json`: class names, plus each model's temperature and conformal thresholds;
* `run_on_device.py`: the benchmark script. It needs only `numpy` and `ai-edge-litert`.

On the device:

```bash
pip install numpy ai-edge-litert
unzip bcdc_edge_bundle.zip && cd bcdc_edge_bundle
python run_on_device.py --threads 4
```

It prints latency, accuracy, glioma sensitivity, ECE and conformal coverage for each model, and writes `device_results.json`. Copy that file back to compare against the Colab numbers.

In [ ]:
bundle = P2 / "bcdc_edge_bundle"
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / "models").mkdir(parents=True)
for (name, mode), path in VARIANTS.items():
    if mode in ("int8", "fp16"):
        shutil.copy(path, bundle / "models" / path.name)
n = min(EDGE_TESTSET_MAX, len(ARR["test"]))
pick = np.sort(np.random.default_rng(SEED).choice(len(ARR["test"]), n, replace=False))
np.savez_compressed(bundle / "testset.npz", images=ARR["test"][pick], labels=Y["test"][pick])
(bundle / "calibration.json").write_text(json.dumps({
    "class_names": CLASS_NAMES, "alpha": ALPHA, "img_size": list(IMG_SIZE), "selected_model": CHOSEN,
    "models": {k: v for k, v in CAL.items() if k.endswith(("_int8", "_fp16"))}}, indent=2))
shutil.copy(Path(bcdc.__file__).parent.parent / "edge" / "run_on_device.py", bundle / "run_on_device.py")
archive = shutil.make_archive(str(P2 / "bcdc_edge_bundle"), "zip", P2, "bcdc_edge_bundle")
print(f"Edge bundle: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")

# Smoke-test the device script here (Colab CPU) on a few images
r = subprocess.run([sys.executable, str(bundle / "run_on_device.py"), "--bundle", str(bundle), "--runs", "5",
                    "--max-images", "40", "--threads", "2"], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])

## E. Federated learning: hospitals train together without sharing scans

We split the training set across **4 simulated hospitals** in two ways:

* **IID:** each hospital gets a random share, so the case mixes are similar.
* **Non-IID:** label proportions are drawn from Dirichlet(α = 0.5), so case mixes differ. For example, one hospital sees mostly gliomas.

All methods use the same model (MobileNetV2, ImageNet start, head plus top 35% trainable, normalisation frozen), the same learning rate and the same total number of passes over the data:

| Method | What it is |
|---|---|
| **Centralised** | All data in one place: the upper bound. |
| **FedAvg** | Each round, every hospital trains 1 local epoch, and the server averages the models weighted by data size. |
| **FedProx** | FedAvg plus a proximal term (μ = 0.01) that limits client drift on non-IID data. |
| **Local only** | Each hospital trains alone on its own data. This is the lower bound, reported as mean ± std over hospitals. |

Every final model is evaluated on the same internal test set. Conformal sets are calibrated on val-B, and temperatures fitted on val-A. Each finished run is saved, so a re-run skips it.

> FedAvg keeps images at each hospital but is **not a privacy guarantee** on its own. Model updates can leak information, so real deployments add secure aggregation and/or differential privacy (e.g. with Flower).

In [ ]:
def fl_model():
    model, base = build_classifier(FL_BACKBONE, NUM_CLASSES, IMG_SIZE, weights=WEIGHTS, seed=SEED)
    set_finetune_trainable(base, TOP_FRACTION)
    return model


def client_ds(idx, seed):
    return labeled_path_dataset(PATHS["train"][idx], Y["train"][idx], IMG_SIZE, BATCH_SIZE, shuffle=True, seed=seed)


def final_metrics(model):
    p = {s: model.predict(VAL_DS[s], verbose=0) for s in ["valA", "valB", "test"]}
    m, _ = trust_metrics(p["valA"], p["valB"], p["test"])
    return {k: m[k] for k in ["accuracy", "macro_f1", "glioma_sens", "ece", "coverage", "set_size"]}


def run_federated(tag, partition, mu=0.0, rounds=FL_ROUNDS):
    out_path = P2 / "federated" / f"{tag}.json"
    if out_path.exists():
        print(f"{tag}: loaded saved result")
        return json.loads(out_path.read_text())
    t0 = time.time()
    model = fl_model()
    trainer = FederatedTrainer(model, lr=FL_LR, mu=mu)
    clients = [client_ds(idx, SEED + k) for k, idx in enumerate(partition)]
    sizes = [len(idx) for idx in partition]
    curve = []
    for r in range(rounds):
        trainer.run_round(clients, sizes, FL_LOCAL_EPOCHS)
        curve.append(float((model.predict(VAL_DS["valA"], verbose=0).argmax(1) == Y["valA"]).mean()))
        print(f"  {tag} round {r + 1}/{rounds}: val-A accuracy {curve[-1]:.3f}")
    res = {"tag": tag, "curve": curve, **final_metrics(model),
           "communication_mb": 0.0 if len(partition) == 1 else trainer.communication_mb(len(partition), rounds),
           "trainable_params": trainer.n_trainable_params, "minutes": (time.time() - t0) / 60}
    out_path.write_text(json.dumps(res, indent=2))
    return res


y_train = Y["train"]
PARTS = {"IID": iid_partition(y_train, FL_CLIENTS, seed=SEED),
         "non-IID": dirichlet_partition(y_train, FL_CLIENTS, alpha=FL_DIRICHLET_ALPHA, seed=SEED,
                                        min_size=max(10, len(y_train) // (FL_CLIENTS * 20)))}
for kind, parts in PARTS.items():
    mix = pd.DataFrame([[int((y_train[p] == c).sum()) for c in range(NUM_CLASSES)] for p in parts],
                       columns=[display_name(c) for c in CLASS_NAMES], index=[f"hospital {k + 1}" for k in range(len(parts))])
    display(Markdown(f"**{kind} partition** (training images per hospital)\n\n" + md_table(mix, "{:.0f}")))
    REPORT.append(f"**{kind} partition**\n\n" + md_table(mix, "{:.0f}"))

In [ ]:
FL = {}
FL["centralised"] = run_federated("centralised", [np.arange(len(y_train))])
for kind, parts in PARTS.items():
    FL[f"FedAvg ({kind})"] = run_federated(f"fedavg_{kind}", parts, mu=0.0)
FL[f"FedProx μ={FL_PROX_MU} (non-IID)"] = run_federated("fedprox_non-IID", PARTS["non-IID"], mu=FL_PROX_MU)
for kind, parts in PARTS.items():
    locals_ = [run_federated(f"local_{kind}_h{k + 1}", [p]) for k, p in enumerate(parts)]
    keys = ["accuracy", "macro_f1", "glioma_sens", "ece", "coverage", "set_size"]
    FL[f"local only ({kind}, mean)"] = {**{k: float(np.mean([l[k] for l in locals_])) for k in keys},
                                       "accuracy_std": float(np.std([l["accuracy"] for l in locals_])),
                                       "curve": list(np.mean([l["curve"] for l in locals_], axis=0)),
                                       "communication_mb": 0.0}

fl_table = pd.DataFrame({k: {c: v[c] for c in ["accuracy", "macro_f1", "glioma_sens", "ece", "coverage", "set_size",
                                                "communication_mb"]} for k, v in FL.items()}).T
display(Markdown(f"**Federated learning** ({FL_CLIENTS} hospitals, {FL_ROUNDS} rounds × {FL_LOCAL_EPOCHS} local epoch; "
                 f"internal test set)\n\n" + md_table(fl_table)))
REPORT.append("### E. Federated learning (internal test set)\n\n" + md_table(fl_table))

plt.figure(figsize=(7, 4))
for k, v in FL.items():
    plt.plot(range(1, len(v["curve"]) + 1), v["curve"], marker=".", label=k)
plt.xlabel("round (= epochs over each hospital's data)"); plt.ylabel("val-A accuracy")
plt.title("Federated vs centralised vs local training"); plt.legend(fontsize=7); plt.tight_layout()
plt.savefig(P2 / "figures" / "federated_curves.png", dpi=120); plt.show()

## 3. Save the Phase 2 results

In [ ]:
header = (f"## Phase 2 results\n\nInternal test set: {len(Y['test'])} images · image size {IMG_SIZE[0]}×{IMG_SIZE[1]} · "
          f"latency measured on the Colab CPU (batch 1), not on an edge device\n")
(P2 / "results_phase2.md").write_text(header + "\n" + "\n\n".join(REPORT) + "\n")
print("Saved", P2 / "results_phase2.md", "| edge bundle:", P2 / "bcdc_edge_bundle.zip")

## 4. Limitations of Phase 2

* **Colab CPU is not an edge device.** These latencies only rank the models. Real Raspberry Pi or Jetson numbers come from `run_on_device.py`.
* **One training seed per backbone.** Differences of about 1 percentage point between backbones may be noise; Phase 0 shows the seed-to-seed spread for MobileNetV2.
* **EfficientNetB0 stands in for EfficientNet-Lite**, which isn't in `keras.applications`. Lite drops Swish and squeeze-excitation to be friendlier to INT8, so its INT8 results may be better.
* **The federated runs are a single-machine simulation.** Hospitals are created by splitting one dataset, so real between-hospital differences (scanner, protocol) are only approximated by label skew. There's no secure aggregation or differential privacy.
* **Everything is evaluated on the internal test set.** The external Figshare check from Phase 1 can be repeated for the chosen edge model.